# Highest-Grossing Films — Data Wrangling Project

In this notebook, I document the full workflow of my project for the Data Wrangling and Visualization assignment. I use the Wikipedia page on highest-grossing films as the main source, parse the data, clean it, enrich missing details such as directors and countries, load the final dataset into a relational database, and then export it to JSON for the frontend published on GitHub Pages.

**Primary source:** [List of highest-grossing films](https://en.wikipedia.org/wiki/List_of_highest-grossing_films)


## How this notebook covers the assignment

According to the assignment, the notebook should show a reproducible parsing workflow, a cleaned dataset prepared for a relational database, and clear documentation of the whole process. In my project, I organized the work in the following order:

1. environment setup and project imports;
2. parsing the source Wikipedia page;
3. cleaning the extracted data and checking its quality;
4. enriching the dataset with director and country information;
5. loading the final dataset into PostgreSQL;
6. exporting JSON for the static frontend on GitHub Pages;
7. briefly exploring the final dataset to make sure the results look correct.


In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import settings
from src.wikipedia_parser import load_html, parse_highest_grossing_table, build_dataset
from src.database import get_connection, initialize_schema, upsert_films
from src.pipeline import export_json

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

## Project configuration

To keep the notebook consistent with the rest of the repository, I use the shared configuration from `src/config.py`. This makes the notebook part of the same project pipeline instead of a separate one-off script.

The configuration file stores the main project settings, including:

- the Wikipedia source URL,
- the path to the local HTML snapshot,
- the PostgreSQL connection parameters,
- the JSON export path used by the frontend.


In [ ]:
config_summary = pd.Series(
    {
        'Wikipedia source': settings.wikipedia_source_url,
        'Raw HTML path': str(settings.raw_html_path),
        'Frontend JSON path': str(settings.site_json_path),
        'PostgreSQL host': settings.postgres_host,
        'PostgreSQL database': settings.postgres_db,
        'Request timeout (seconds)': settings.request_timeout,
    },
    name='value',
)
config_summary

## 1. Load the source HTML

At this stage, I load the HTML of the source page. If a local snapshot already exists, the notebook uses it; otherwise, it downloads the page. I chose this approach because it makes development more reproducible and avoids sending repeated requests to the same Wikipedia page while I test the parser.


In [ ]:
html = load_html()
print(f'Loaded {len(html):,} characters from the source page.')
print(f'Using snapshot: {settings.raw_html_path.exists()}')

## 2. Parse the main Wikipedia table

The first parser reads the main table from the Wikipedia page and extracts the fields that are directly available there:

- source rank,
- film title,
- release year,
- box office value as text,
- cleaned numeric box office value,
- link to the film's individual Wikipedia page.

At this point, the dataset is still incomplete, because the director and country fields are not reliably stored in the main list table. I add them in the next step by visiting each film page separately.


In [ ]:
base_records = parse_highest_grossing_table(html)
base_df = pd.DataFrame(base_records)

print(f'Parsed {len(base_df)} rows from the main Wikipedia table.')
display(base_df.head(10))

## 3. Initial data quality checks

Before I enrich the dataset, I first check whether the base parsing step worked correctly. Here I look at missing values, blank strings, duplicates, and column types. This helps me catch problems early and make sure the structure is stable before moving to the next stage.


In [ ]:
base_quality = pd.DataFrame(
    {
        'missing_values': base_df.isna().sum(),
        'blank_strings': [(base_df[col].astype(str).str.strip() == '').sum() for col in base_df.columns],
        'dtype': base_df.dtypes.astype(str),
    },
    index=base_df.columns,
)

duplicate_titles = base_df.duplicated(subset=['title', 'release_year']).sum()

display(base_quality)
print(f'Duplicate (title, release_year) pairs: {duplicate_titles}')
print(f'Earliest parsed year: {base_df.release_year.min()}')
print(f'Latest parsed year: {base_df.release_year.max()}')

## 4. Enrich records from film detail pages

The assignment requires information about the director and the country of origin, but these fields are not consistently available in the summary table. To solve this, my project opens each film's Wikipedia page and reads the needed values from the infobox.

During development, I can temporarily set `FULL_DATASET_LIMIT = 10` to test the pipeline faster. For the final version, it stays `None` so the full dataset is built.

The code below first tries to run the full enrichment workflow from Wikipedia pages. If external requests are unavailable, it falls back to the already exported JSON file so that the notebook still works from start to finish.


In [ ]:
FULL_DATASET_LIMIT = None

try:
    dataset = build_dataset(limit=FULL_DATASET_LIMIT)
    records = [record.to_dict() for record in dataset]
    dataset_status = 'Built enriched dataset from Wikipedia detail pages.'
except Exception as exc:
    fallback_path = project_root / 'docs' / 'data' / 'films.json'
    records = json.loads(fallback_path.read_text(encoding='utf-8'))
    dataset_status = f'Fell back to cached JSON because live enrichment failed: {exc}'

films_df = pd.DataFrame(records)

print(dataset_status)
print(f'Working dataset contains {len(films_df)} films.')
display(films_df.head(10))

## 5. Wrangling the enriched dataset

After enrichment, I prepare the dataset for validation and analysis. Most of the raw HTML noise is already removed during parsing, but I still create a few helper columns that make the data easier to interpret:

- `box_office_billions` for simpler numeric comparison,
- `decade` for grouping films by time period,
- `country_count` to identify films associated with multiple countries,
- `director_count` to identify films with multiple directors.


In [ ]:
analysis_df = films_df.copy()
analysis_df['box_office_billions'] = analysis_df['box_office_value'] / 1_000_000_000
analysis_df['decade'] = (analysis_df['release_year'] // 10) * 10
analysis_df['country_count'] = analysis_df['country'].fillna('').apply(
    lambda value: len([part.strip() for part in value.split(',') if part.strip()])
)
analysis_df['director_count'] = analysis_df['director'].fillna('').apply(
    lambda value: len([part.strip() for part in value.split(',') if part.strip()])
)

analysis_df[['title', 'release_year', 'director', 'country', 'box_office_billions', 'decade']].head(10)

## 6. Final quality checks

Now I run one more validation step on the enriched dataset. The goal here is to confirm that the final table matches the assignment requirements and is ready to be inserted into the relational database without duplicate or incomplete key records.


In [ ]:
final_quality = pd.DataFrame(
    {
        'missing_values': analysis_df.isna().sum(),
        'unknown_director_rows': [0] * len(analysis_df.columns),
        'unknown_country_rows': [0] * len(analysis_df.columns),
    },
    index=analysis_df.columns,
)
final_quality.loc['director', 'unknown_director_rows'] = (analysis_df['director'] == 'Unknown').sum()
final_quality.loc['country', 'unknown_country_rows'] = (analysis_df['country'] == 'Unknown').sum()

display(final_quality)
print('Duplicate final keys:', analysis_df.duplicated(subset=['title', 'release_year']).sum())
print('Rows with missing Wikipedia URL:', analysis_df['wikipedia_url'].isna().sum() + (analysis_df['wikipedia_url'].fillna('').str.strip() == '').sum())

## 7. Analytical summary

The main presentation of this project is the website, but I also include a short analytical summary in the notebook. This helps me verify that the extracted data looks reasonable and gives a quick overview of patterns such as top-grossing films, decade trends, and country frequency.


In [ ]:
top_grossing = analysis_df.nlargest(10, 'box_office_value')[
    ['title', 'release_year', 'director', 'country', 'box_office_text']
]

decade_summary = (
    analysis_df.groupby('decade', dropna=True)
    .agg(
        film_count=('title', 'count'),
        average_box_office=('box_office_value', 'mean'),
        max_box_office=('box_office_value', 'max'),
    )
    .sort_index()
)

country_frequency = (
    analysis_df['country']
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
    .rename_axis('country')
    .reset_index(name='films')
)

print('Top 10 films by box office')
display(top_grossing)
print('Decade summary')
display(decade_summary)
print('Most common countries in the dataset')
display(country_frequency)

## 8. Optional notebook charts

Since the assignment also focuses on visualization, I added a couple of simple charts inside the notebook. They are not the main frontend visuals, but they are useful for a quick check of the processed data. If `matplotlib` is unavailable in the environment, this step is skipped safely.


In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    analysis_df.nlargest(10, 'box_office_value').sort_values('box_office_value').plot(
        kind='barh',
        x='title',
        y='box_office_billions',
        legend=False,
        ax=axes[0],
        color='#1f77b4',
        title='Top 10 films by box office (billions USD)',
    )
    axes[0].set_xlabel('Box office (billions USD)')
    axes[0].set_ylabel('Film title')

    analysis_df['decade'].value_counts().sort_index().plot(
        kind='bar',
        ax=axes[1],
        color='#ff7f0e',
        title='Film count by decade',
    )
    axes[1].set_xlabel('Decade')
    axes[1].set_ylabel('Number of films')

    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print('matplotlib is not installed, so the plotting step was skipped.')

## 9. Database design

The assignment asks for storing the parsed data in a relational database. In my project, I use PostgreSQL and keep the structure close to the required schema, while also adding a few practical fields that help with traceability and updates:

- `source_rank` keeps the original order from the Wikipedia table,
- `wikipedia_url` stores the source link for each film,
- `extracted_at` and `updated_at` make the loading process easier to track.


In [ ]:
schema_path = project_root / 'db' / 'schema.sql'
print(schema_path.read_text(encoding='utf-8'))

## 10. Load cleaned data into PostgreSQL

In this step, I create the table if it does not already exist and then load the cleaned records into PostgreSQL. The notebook uses an upsert based on `(title, release_year)`, so rerunning the pipeline updates existing entries instead of inserting duplicates.

If PostgreSQL is not running, the notebook will show the connection error, but the parsing and JSON export parts can still be reviewed independently.


In [ ]:
db_status = {}

try:
    with get_connection(settings) as connection:
        initialize_schema(connection)
        rows_loaded = upsert_films(connection, records)
        total_rows = connection.execute('SELECT COUNT(*) FROM films').fetchone()[0]

    db_status = {
        'rows_loaded_this_run': rows_loaded,
        'rows_currently_in_database': total_rows,
        'status': 'success',
    }
except Exception as exc:
    db_status = {
        'rows_loaded_this_run': None,
        'rows_currently_in_database': None,
        'status': f'skipped: {exc}',
    }

pd.Series(db_status, name='database_load')

## 11. Export JSON for the GitHub Pages frontend

Because GitHub Pages only supports static hosting, the frontend cannot connect directly to PostgreSQL. For that reason, I export the cleaned dataset to a JSON file, and the website reads this file on the client side with JavaScript.


In [ ]:
output_path = export_json(records, project_root / 'docs' / 'data' / 'films.json')
print(f'Exported JSON to: {output_path}')
print(f'File exists: {output_path.exists()}')
print(f'File size: {output_path.stat().st_size:,} bytes')

## 12. Submission checklist

This notebook covers the Python side of my project and the main technical requirements of the assignment:

- parsing the Wikipedia source page,
- cleaning and validating the extracted data,
- preparing the final relational dataset,
- loading the data into PostgreSQL,
- exporting JSON for the website.

For the final submission, the three deliverables should be submitted together:

1. the GitHub Pages link to the deployed website;
2. the GitHub repository with the frontend and ETL code;
3. this Jupyter Notebook.
